# Hyperparameter search

A model's **parameters** are learned from the data by `fit()` (the coefficients of a linear regression, the splits of a tree). Its **hyperparameters** are the settings we choose *before* fitting (the number of trees in a forest, the depth of each tree, the imputation strategy of a preprocessing step) and they can change performance a lot.

To avoid the tedious process of manually modifying a model's hyperparameters until finding the ones that yield the best results, we can define all the hyperparameter values we want to test and program them to try many combinations, each evaluated with [cross-validation](cross_validation.ipynb) so that the test set is never touched (see [Data leakage](data_leakage.ipynb)).

## Loading the data and building a pipeline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

housing = pd.read_csv("../data/housing.csv")
train_set, test_set = train_test_split(
    housing, test_size=0.2, random_state=42,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]))

X_train = train_set.drop("median_house_value", axis=1)  # features
y_train = train_set["median_house_value"].copy()        # target (labels)

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

preprocessing = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include=np.number)),
    ("cat", OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include=object)),
])

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])
full_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('random_forest', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

## Viewing hyperparameters

Every hyperparameter of every step of a pipeline is addressable with the `<step>__<hyperparameter>` naming convention, nested as deep as needed. To see all the names that can be tuned:

In [ ]:
for param in sorted(full_pipeline.get_params().keys()):
    print(param)

memory
preprocessing
preprocessing__cat
preprocessing__cat__categories
preprocessing__cat__drop
preprocessing__cat__dtype
preprocessing__cat__feature_name_combiner
preprocessing__cat__handle_unknown
preprocessing__cat__max_categories
preprocessing__cat__min_frequency
preprocessing__cat__sparse_output
preprocessing__force_int_remainder_cols
preprocessing__n_jobs
preprocessing__num
preprocessing__num__memory
preprocessing__num__simpleimputer
preprocessing__num__simpleimputer__add_indicator
preprocessing__num__simpleimputer__copy
preprocessing__num__simpleimputer__fill_value
preprocessing__num__simpleimputer__keep_empty_features
preprocessing__num__simpleimputer__missing_values
preprocessing__num__simpleimputer__strategy
preprocessing__num__standardscaler
preprocessing__num__standardscaler__copy
preprocessing__num__standardscaler__with_mean
preprocessing__num__standardscaler__with_std
preprocessing__num__steps
preprocessing__num__transform_input
preprocessing__num__verbose
preprocessing__

## Grid search

`GridSearchCV` tries **every combination** of the values we list. The `param_grid` parameter is a list of dictionaries, each containing the hyperparameter values we want to test; the combinations of each dictionary are added up. Here the first dictionary tests 2 imputation strategies × 3 values of `max_features` (the number of features considered at each split), and the second 2 values of `max_features` × 2 values of `n_estimators` (the number of trees): 2×3 + 2×2 = 10 combinations, each evaluated with 3-fold cross-validation, i.e. 30 trainings.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"preprocessing__num__simpleimputer__strategy": ["median", "mean"],  # a preprocessing hyperparameter
     "random_forest__max_features": [4, 6, 8]},  # features considered when looking for the best split
    {"random_forest__max_features": [6, 8],
     "random_forest__n_estimators": [50, 200]},  # number of trees
]
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

_ = grid_search.fit(X_train, y_train)

The `n_jobs` parameter allows parallelizing the hyperparameter search by indicating the number of processors to use; a value of -1 means all available processors will be used. This same parameter can be used in the `RandomForestRegressor` model to parallelize tree construction, but you need to be careful if doing both, since if you parallelize each hyperparameter search, which is itself a model execution, and that model in turn parallelizes tree construction, the total number of executions would multiply. The total `n_jobs` of `RandomForestRegressor` multiplied by the number of searches cannot exceed the number of physical cores on the machine. In general, it's better to parallelize the hyperparameter search rather than tree construction; therefore, in this case, we've chosen to leave `RandomForestRegressor` with the default value `n_jobs=None`, which assigns 1 core per model, and use the maximum for `GridSearchCV` with `n_jobs=-1`.

We can now see the best hyperparameters found:

In [ ]:
grid_search.best_params_

{'random_forest__max_features': 6, 'random_forest__n_estimators': 200}

It also returns the best estimator found, already refitted on the whole training set:

In [ ]:
grid_search.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('random_forest', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

We can also see the result of each hyperparameter combination tested during the search:

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

# Select the columns we want to display
cv_res = cv_res[["param_preprocessing__num__simpleimputer__strategy", "param_random_forest__max_features",
                 "param_random_forest__n_estimators",
                 "split0_test_score", "split1_test_score", "split2_test_score", "mean_test_score"]]

# Rename columns for simplicity
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]
cv_res.columns = ["imputer_strategy", "max_features", "n_estimators"] + score_cols
# Scores are negative RMSEs: flip the sign and round
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)
cv_res

,imputer_strategy,max_features,n_estimators,split0,split1,split2,mean_test_rmse
7,NaN,6,200.0,48696,49855,50498,49683
9,NaN,8,200.0,48825,49886,50587,49766
4,mean,6,NaN,48880,49846,50642,49789
2,median,8,NaN,48945,49898,50621,49821
1,median,6,NaN,48970,49955,50563,49829
5,mean,8,NaN,48840,50097,50605,49848
6,NaN,6,50.0,49358,50033,50816,50069
8,NaN,8,50.0,49274,50288,50810,50124
0,median,4,NaN,48846,50760,51009,50205
3,mean,4,NaN,49221,50504,51094,50273


When the best value of a hyperparameter is the highest (or lowest) one tested, it makes sense to run new tests with values beyond it: the optimum may lie outside the grid.

## Randomized search

Instead of testing all possible hyperparameter combinations, `RandomizedSearchCV` allows testing a specified number of random combinations. This provides certain advantages:

- **Computational efficiency**: The "curse of dimensionality" makes grid search computationally infeasible very quickly. With more than 3 or 4 hyperparameters with a few options each, the total number of combinations to test explodes. Randomized search allows setting a computational budget (number of iterations) independent of the number of hyperparameters, making it feasible for complex problems.

- **Effectiveness in high dimensions**: For many objective functions (such as model performance), only a few hyperparameters have a significant impact. By randomly sampling combinations, there's a higher probability of testing diverse values in the important dimensions, while grid search wastes much effort systematically testing values in dimensions that barely affect the result.

- **Handling continuous parameters**: Randomized search naturally handles continuous parameters by sampling from a distribution (e.g., uniform, log-uniform). Grid search requires discretizing the range, which is artificial and can easily miss the actual optimal value if it falls between grid points.

For these reasons, `RandomizedSearchCV` is generally more efficient than `GridSearchCV` for high-dimensional problems, with many hyperparameters, or where we don't have a clear idea of the hyperparameter value ranges.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    "random_forest__max_features": randint(low=2, high=10),  # any integer from 2 to 9
    "random_forest__n_estimators": randint(low=50, high=300),
    "random_forest__min_samples_leaf": randint(low=1, high=10),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,  # number of combinations to try
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)

_ = rnd_search.fit(X_train, y_train)

`scipy.stats.randint()` returns an object containing the probability distribution of a discrete random variable (`uniform` and `loguniform` are the continuous counterparts). `RandomizedSearchCV` uses it to randomly sample hyperparameter values. The `n_iter` parameter is the number of combinations to try: here 10, so the search costs the same as the grid search above while exploring three hyperparameters instead of two.

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_res = cv_res[["param_random_forest__max_features", "param_random_forest__n_estimators",
                 "param_random_forest__min_samples_leaf",
                 "split0_test_score", "split1_test_score", "split2_test_score", "mean_test_score"]]
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]
cv_res.columns = ["max_features", "n_estimators", "min_samples_leaf"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)
cv_res

,max_features,n_estimators,min_samples_leaf,split0,split1,split2,mean_test_rmse
0,8,142,4,49600,50605,51402,50536
9,9,210,5,49975,50874,51714,50854
8,7,179,6,50329,51490,52205,51341
2,6,171,7,50539,51963,52599,51700
1,8,238,8,50821,52201,52707,51910
6,4,102,6,51054,52670,53329,52351
5,5,201,8,51173,52894,53303,52457
3,4,124,7,51150,52965,53392,52502
4,4,166,8,51499,53214,53782,52831
7,3,285,8,52498,54529,55038,54022


## Iterating: narrowing the search

A single randomized search rarely finds the optimum. The usual practice is to run successive iterations: fix the hyperparameters for which all the best results have converged to a single value, and define a narrower range centered around the best results for the rest. For a random forest, the hyperparameters that most often matter are:

| Hyperparameter      | Description |
|---------------------|-------------|
| `n_estimators`     | Number of trees in the forest. More trees can improve accuracy but increase computation time. |
| `max_depth`        | Maximum depth of each tree. A low value may lead to *underfitting*, while a high value may lead to *overfitting*. |
| `max_features`     | Number of *features* considered at each split. Can be an integer, a percentage, `"sqrt"` or `"log2"`. Fewer *features* can reduce variance (and thus *overfitting*). |
| `min_samples_split` | Minimum number of samples required to split a node. Higher values reduce *overfitting*. |
| `min_samples_leaf`  | Minimum number of samples in a leaf. Higher values smooth the prediction. |
| `max_samples`      | Percentage of samples used in each tree. Useful for reducing *overfitting*. |

Preprocessing steps have hyperparameters too (the imputation strategy above; in the California pipeline, the number of geographic clusters and the `gamma` of the similarity kernel), and searching them together with the model's is one of the practical reasons to keep everything in a single pipeline. The two-iteration search of the California project is in [`../workflow/07_hyperparameter_tuning.ipynb`](../workflow/07_hyperparameter_tuning.ipynb).

`%%time` at the top of a cell is a [Jupyter magic command](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-time) that measures the execution time of the cell, useful to decide how big the next iteration can be.

## Evaluating the final model

Once the search is over, `best_estimator_` is the model to keep. Its expected performance is estimated **once** on the test set; using the test set to choose between candidates would be [model selection leakage](data_leakage.ipynb#Model-selection-leakage).

In [ ]:
from sklearn.metrics import root_mean_squared_error

X_test = test_set.drop("median_house_value", axis=1)
y_test = test_set["median_house_value"]

final_predictions = rnd_search.best_estimator_.predict(X_test)
root_mean_squared_error(y_test, final_predictions)

46960.93112433843